# Phase 3: Figure Parser

This notebook demonstrates the `FigureParser` — given a `Paper` object (from Phase 2),
extracting fully structured `Figure` objects with subfigures, axes, data sources, and purpose.

**What this notebook covers:**
1. Caption extraction (regex from section text or JATS `figure_captions` dict)
2. In-text reference collection (sentences that cite the figure)
3. Dataset accession extraction (regex-first, LLM fallback)
4. Subfigure decomposition via LLM (`SubFigure` objects with `PlotType`, `Axis`, `ColorMapping`)
5. Full `parse_figure()` and `parse_all_figures()` calls
6. Panel layout inference
7. JSON round-trip verification
8. Graceful failure path (stub figure)

**Prerequisites:** Set `ANTHROPIC_API_KEY` in your environment for LLM-assisted steps
(subfigure decomposition, purpose extraction, method extraction). Caption finding,
in-text reference collection, and dataset accession extraction are regex-only and
work without an API key.

In [ ]:
import json, os

from researcher_ai.models.figure import (
    Figure, SubFigure, Axis, AxisScale,
    PlotType, PlotCategory, PlotLayer,
    ColorMapping, ColormapType,
    ErrorBarType, StatisticalAnnotation,
    PanelLayout, RenderingSpec,
)
from researcher_ai.models.paper import Paper, Section, PaperSource, PaperType
from researcher_ai.parsers.figure_parser import (
    FigureParser,
    _fig_ids_match,
    _build_fig_ref_pattern,
    _extract_caption_from_text,
    _subfigure_from_meta,
    _axis_from_meta,
    _SubFigureMeta, _AxisMeta,
)
from researcher_ai.utils.llm import LLMCache

# Use a local cache to avoid repeated API calls during development
CACHE_DIR = ".llm_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

print('researcher_ai.parsers.figure_parser imported successfully')

## 1. Caption extraction

`_find_caption()` searches in this order:
1. `paper.figure_captions` dict (populated from JATS XML — richest source)
2. Paragraph-level regex scan across all sections
3. Raw text scan as final fallback

In [ ]:
# Caption normalisation: both sides are normalised before comparison
test_pairs = [
    ("Figure 1",              "Figure 1"),
    ("Fig. 1",               "Figure 1"),
    ("FIGURE 2",             "figure 2"),
    ("Supplementary Figure S1", "Supplementary Figure S1"),
    ("Figure 1A",            "Figure 2"),   # should not match
]

for key, fig_id in test_pairs:
    match = _fig_ids_match(key, fig_id)
    print(f"  {key!r:<30} vs {fig_id!r:<30} → {'✓ match' if match else '✗ no match'}")

In [ ]:
# Regex scan for captions in raw text
raw_text = """
The eCLIP protocol is described in Figure 1.

Figure 1. Overview of the eCLIP pipeline.
(a) Schematic of library preparation steps.
(b) Volcano plot of enrichment (log2FC vs. -log10 p-value) for all binding peaks.
n=3 biological replicates. Error bars represent SEM.
(c) UMAP embedding of 10,000 cells coloured by cluster identity.

Figure 2. ChIP-seq peak calling results.
Peaks called with MACS2 are shown for all 150 RBPs.
"""

for fig_id in ["Figure 1", "Figure 2", "Figure 3"]:
    caption = _extract_caption_from_text(raw_text, fig_id)
    print(f"\n{fig_id}:")
    print(f"  {caption[:120].strip()!r}{'...' if len(caption) > 120 else ''}")

In [ ]:
# Build a minimal Paper and call _find_caption() directly
paper_with_jats = Paper(
    title="Test paper",
    authors=["Author A"],
    abstract="Abstract text.",
    source=PaperSource.PMCID,
    source_path="PMC0000000",
    paper_type=PaperType.EXPERIMENTAL,
    sections=[],
    figure_ids=["Figure 1", "Figure 2"],
    figure_captions={
        "Figure 1": "JATS caption for Figure 1: full text with panel descriptions.",
        "Fig. 2":   "JATS caption for Figure 2: abbreviated label.",
    },
)

parser = FigureParser(cache_dir=CACHE_DIR)

for fig_id in ["Figure 1", "Figure 2"]:
    caption = parser._find_caption(paper_with_jats, fig_id)
    print(f"{fig_id}: {caption!r}")

## 2. In-text reference collection

`_find_in_text_references()` collects every sentence across all sections that
mentions the figure ID or its sub-panel variants (e.g., `Figure 1a`, `Fig. 1B-D`).

In [ ]:
# Build the figure reference pattern and test it
test_sentences = [
    "The eCLIP protocol (Figure 1a) enables transcriptome-wide mapping.",
    "See Figure 1b for the volcano plot summary.",
    "UMAP clustering is presented in Figure 1c.",
    "ChIP-seq peaks are shown in Figure 2.",
    "This sentence has no figure reference.",
    "Figure 1A-D provides a comprehensive overview.",
    "Supplementary Fig. S1 shows QC metrics.",
]

pattern = _build_fig_ref_pattern("Figure 1")
supp_pattern = _build_fig_ref_pattern("Supplementary Figure S1")

print("Figure 1 reference matches:")
for s in test_sentences:
    matched = "✓" if pattern.search(s) else "✗"
    supp_matched = "✓" if supp_pattern.search(s) else "✗"
    print(f"  Fig1={matched} SuppS1={supp_matched}  {s[:70]!r}")

In [ ]:
# Use a richer Paper to show in-text reference collection in action
sample_sections = [
    Section(
        title="Introduction",
        text=(
            "The eCLIP protocol (Figure 1a) enables transcriptome-wide mapping "
            "of RNA–protein interactions. See Figure 1b for the volcano plot summary. "
            "ChIP-seq peaks are shown in Figure 2."
        ),
        figures_referenced=["Figure 1", "Figure 2"],
    ),
    Section(
        title="Results",
        text=(
            "We applied eCLIP to 150 RBPs. Figure 1b shows significantly enriched peaks. "
            "UMAP clustering is presented in Figure 1c. Table S1 lists all peaks."
        ),
        figures_referenced=["Figure 1"],
    ),
    Section(
        title="Methods",
        text=(
            "RNA-seq libraries were prepared from HEK293 cells. "
            "Peak calling was performed with CLIPper v2.0. "
            "Differential enrichment was computed with DESeq2."
        ),
        figures_referenced=[],
    ),
]

sample_paper = Paper(
    title="Robust transcriptome-wide discovery of RNA-binding protein binding sites",
    authors=["Van Nostrand, EL", "Yeo, GW"],
    abstract="eCLIP is a method for high-throughput CLIP-seq.",
    doi="10.1038/nmeth.3810",
    pmid="26971820",
    pmcid="PMC4878918",
    source=PaperSource.PMCID,
    source_path="PMC4878918",
    paper_type=PaperType.EXPERIMENTAL,
    sections=sample_sections,
    figure_ids=["Figure 1", "Figure 2"],
    raw_text=(
        "\n".join(s.text for s in sample_sections)
        + "\n\nFigure 1. Overview of the eCLIP pipeline. "
        "(a) Schematic. (b) Volcano plot of enrichment (log2FC, GSE72987). (c) UMAP embedding."
    ),
)

parser = FigureParser(cache_dir=CACHE_DIR)

for fig_id in ["Figure 1", "Figure 2"]:
    refs = parser._find_in_text_references(sample_paper, fig_id)
    print(f"\n{fig_id}: {len(refs)} references found")
    for r in refs:
        print(f"  • {r[:90]}")

## 3. Dataset accession extraction

`_identify_datasets()` uses a regex pass first — reliable for well-formatted
accessions (GEO, SRA, PRIDE, BioProject, ArrayExpress, EGA).
If no accessions are found, it falls back to an LLM call.

In [ ]:
accession_tests = [
    ("Data deposited at GEO under accession GSE72987.",                          []),
    ("Raw reads available at SRA (SRP123456).",                                  []),
    ("Proteomics data at PRIDE: PXD001234.",                                     []),
    ("RNA-seq: GSE12345. ChIP-seq: GSE67890.",                                   []),
    ("ATAC-seq data at EGA (EGAS00001002456).",                                  []),
    ("ArrayExpress: E-MTAB-3929.",                                               []),
    ("BioProject PRJNA485501 contains the raw FASTQ files.",                     []),
    ("Reads in SRR9876543 and SRX4321001.",                                      []),
    ("No repository accession mentioned in this caption.",                       []),  # LLM fallback
]

for caption, in_text in accession_tests:
    # Monkey-patch LLM fallback to avoid real API calls for the empty case
    from unittest.mock import patch
    from researcher_ai.parsers.figure_parser import _MethodsAndDatasets
    with patch("researcher_ai.parsers.figure_parser.ask_claude_structured",
               return_value=_MethodsAndDatasets(methods=[], datasets=["(LLM-fallback-would-run)"])):
        result = parser._identify_datasets(caption, in_text)
    print(f"  {caption[:55]!r:<58} → {result}")

## 4. SubFigure model: pure construction

The `_subfigure_from_meta()` helper converts LLM-extracted `_SubFigureMeta`
into fully typed `SubFigure` objects — no API call needed here.

In [ ]:
# Build representative panels from the eCLIP Figure 1 caption
panel_metas = [
    _SubFigureMeta(
        label="a",
        description="Schematic of the eCLIP library preparation protocol.",
        plot_type="image",
        plot_category="image",
    ),
    _SubFigureMeta(
        label="b",
        description="Volcano plot of enrichment (log2FC) vs. significance (-log10 p-value).",
        plot_type="volcano",
        plot_category="genomic",
        x_axis=_AxisMeta(label="log2FC", scale="log2", data_type="fold change"),
        y_axis=_AxisMeta(label="-log10(p-value)", scale="reversed", is_inverted=True),
        color_variable="significance",
        error_bars="none",
        sample_size="n=3 biological replicates",
        shows_individual_points=True,
        statistical_test="Wilcoxon rank-sum",
        data_source="GSE72987",
    ),
    _SubFigureMeta(
        label="c",
        description="UMAP embedding of 10,000 cells coloured by cluster identity.",
        plot_type="umap",
        plot_category="dimensionality",
        x_axis=_AxisMeta(label="UMAP 1"),
        y_axis=_AxisMeta(label="UMAP 2"),
        color_variable="cluster_identity",
        shows_individual_points=True,
        facet_variable=None,
    ),
]

subfigures = [_subfigure_from_meta(m) for m in panel_metas]

for sf in subfigures:
    print(f"Panel {sf.label}: {sf.plot_type.value} ({sf.plot_category.value})")
    if sf.x_axis:
        print(f"  x_axis: {sf.x_axis.label!r} [{sf.x_axis.scale.value}]")
    if sf.y_axis:
        print(f"  y_axis: {sf.y_axis.label!r} [{sf.y_axis.scale.value}] inverted={sf.y_axis.is_inverted}")
    if sf.color_mapping:
        print(f"  color:  variable={sf.color_mapping.variable!r}")
    if sf.statistical_annotations:
        print(f"  stats:  test={sf.statistical_annotations.test_name!r}")
    print(f"  error_bars={sf.error_bars.value}  individual_pts={sf.shows_individual_points}")
    print()

## 5. Panel layout inference

`_infer_layout()` infers a `PanelLayout` (rows × cols) from the subfigure
labels and count. Labels are also used to detect uppercase vs lowercase style.

In [ ]:
layout_cases = [
    ([],                         "zero panels"),
    (["main"],                   "single panel"),
    (["a", "b"],                 "2 panels, lowercase"),
    (["A", "B", "C"],            "3 panels, uppercase"),
    (["a", "b", "c", "d"],       "4 panels → 2×2"),
    (["a", "b", "c", "d", "e"],  "5 panels → 3×2"),
    (["A", "B", "C", "D", "E", "F"], "6 panels → 2×3 or 3×2"),
]

for labels, desc in layout_cases:
    sfs = [
        SubFigure(
            label=l,
            description="test",
            plot_category=PlotCategory.RELATIONAL,
            plot_type=PlotType.SCATTER,
        )
        for l in labels
    ]
    layout = parser._infer_layout(sfs)
    print(f"  {desc:<35} → {layout.n_rows}×{layout.n_cols} [{layout.panel_labels_style}]")

## 6. Full parse_figure() — LLM in the loop

`parse_figure()` chains caption extraction, in-text reference collection,
and three LLM calls (subfigure decomposition, purpose, methods/datasets).

**Requires `ANTHROPIC_API_KEY`** — skip or mock if not set.

In [ ]:
import os
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY not set — skipping live LLM calls.")
    print("Set the key and re-run this cell for a full demo.")
else:
    parser_live = FigureParser(cache_dir=CACHE_DIR)
    fig = parser_live.parse_figure(sample_paper, "Figure 1")

    print(f"Figure ID:  {fig.figure_id}")
    print(f"Title:      {fig.title}")
    print(f"Purpose:    {fig.purpose[:200]}..." if len(fig.purpose) > 200 else f"Purpose:    {fig.purpose}")
    print(f"Subfigures: {len(fig.subfigures)}")
    for sf in fig.subfigures:
        print(f"  Panel {sf.label}: {sf.plot_type.value} — {sf.description[:60]}")
    print(f"Datasets:   {fig.datasets_used}")
    print(f"Methods:    {fig.methods_used}")
    print(f"Layout:     {fig.layout.n_rows}×{fig.layout.n_cols} [{fig.layout.panel_labels_style}]")
    print(f"In-text:    {len(fig.in_text_context)} sentences")

## 7. Mocked parse_figure() — no API key required

The same code path exercised with mocked LLM responses, so the notebook
is fully runnable without `ANTHROPIC_API_KEY`.

In [ ]:
from unittest.mock import patch
from researcher_ai.parsers.figure_parser import (
    _SubFigureList, _FigurePurpose, _MethodsAndDatasets,
)

MOCK_SUBFIGS = _SubFigureList(subfigures=[
    _SubFigureMeta(
        label="a",
        description="Schematic of the eCLIP protocol.",
        plot_type="image",
        plot_category="image",
    ),
    _SubFigureMeta(
        label="b",
        description="Volcano plot: log2FC vs. -log10(p-value) for binding peaks.",
        plot_type="volcano",
        plot_category="genomic",
        x_axis=_AxisMeta(label="log2FC", scale="log2"),
        y_axis=_AxisMeta(label="-log10(p)", scale="reversed", is_inverted=True),
        error_bars="sem",
        sample_size="n=3 biological replicates",
        statistical_test="Wilcoxon rank-sum",
    ),
    _SubFigureMeta(
        label="c",
        description="UMAP of 10,000 cells coloured by cluster identity.",
        plot_type="umap",
        plot_category="dimensionality",
        x_axis=_AxisMeta(label="UMAP 1"),
        y_axis=_AxisMeta(label="UMAP 2"),
        color_variable="cluster_identity",
        shows_individual_points=True,
    ),
])

MOCK_PURPOSE = _FigurePurpose(
    title="eCLIP pipeline overview and peak enrichment",
    purpose=(
        "Figure 1 introduces the eCLIP experimental workflow and summarises the "
        "transcriptome-wide binding landscape of 150 RNA-binding proteins. "
        "Panel (a) illustrates the library preparation protocol, panel (b) displays "
        "the overall enrichment for all detected binding peaks, and panel (c) shows "
        "the single-cell UMAP clustering used to stratify the dataset."
    ),
)

MOCK_METHODS = _MethodsAndDatasets(
    methods=["eCLIP", "RNA-seq", "CLIPper peak calling", "DESeq2"],
    datasets=["GSE72987"],
)


def _mock_llm(prompt, output_schema, **kw):
    if output_schema is _SubFigureList:
        return MOCK_SUBFIGS
    elif output_schema is _FigurePurpose:
        return MOCK_PURPOSE
    elif output_schema is _MethodsAndDatasets:
        return MOCK_METHODS
    raise ValueError(f"Unexpected schema: {output_schema}")


with patch("researcher_ai.parsers.figure_parser.ask_claude_structured", side_effect=_mock_llm):
    parser_mock = FigureParser()
    fig = parser_mock.parse_figure(sample_paper, "Figure 1")

print(f"Figure ID:  {fig.figure_id}")
print(f"Title:      {fig.title}")
print(f"Purpose:\n  {fig.purpose}")
print(f"\nSubfigures ({len(fig.subfigures)}):")
for sf in fig.subfigures:
    print(f"  [{sf.label}] {sf.plot_type.value:20s}  {sf.description[:55]}")
    if sf.x_axis:
        print(f"       x={sf.x_axis.label!r} [{sf.x_axis.scale.value}]")
    if sf.y_axis:
        print(f"       y={sf.y_axis.label!r} [{sf.y_axis.scale.value}]")
    if sf.color_mapping:
        print(f"       color={sf.color_mapping.variable!r}")
    if sf.statistical_annotations:
        print(f"       stats={sf.statistical_annotations.test_name!r}")
    print(f"       error_bars={sf.error_bars.value}  individual_pts={sf.shows_individual_points}")
print(f"\nDatasets:   {fig.datasets_used}")
print(f"Methods:    {fig.methods_used}")
print(f"Layout:     {fig.layout.n_rows}×{fig.layout.n_cols} [{fig.layout.panel_labels_style}]")
print(f"In-text:    {len(fig.in_text_context)} sentences")

## 8. parse_all_figures() — batch parsing

In [ ]:
with patch("researcher_ai.parsers.figure_parser.ask_claude_structured", side_effect=_mock_llm):
    parser_batch = FigureParser()
    figures = parser_batch.parse_all_figures(sample_paper)

print(f"Parsed {len(figures)} figure(s) from paper with {len(sample_paper.figure_ids)} figure_ids.")
for fig in figures:
    print(f"  {fig.figure_id}: {len(fig.subfigures)} subfigures, "
          f"datasets={fig.datasets_used}, methods={fig.methods_used}")

## 9. Graceful failure: stub figure

When LLM calls fail for a figure, `parse_all_figures()` catches the exception
and returns a minimal stub rather than aborting the entire batch.

In [ ]:
call_count = [0]

def _failing_then_ok(prompt, output_schema, **kw):
    call_count[0] += 1
    if call_count[0] <= 3:   # first 3 calls (subfig + purpose + methods for Figure 1)
        raise RuntimeError(f"Simulated LLM failure (call {call_count[0]})")
    return _mock_llm(prompt, output_schema, **kw)

call_count[0] = 0
with patch("researcher_ai.parsers.figure_parser.ask_claude_structured",
           side_effect=_failing_then_ok):
    parser_fail = FigureParser()
    figures_fail = parser_fail.parse_all_figures(sample_paper)

print(f"Returned {len(figures_fail)} figure(s) despite LLM failure on Figure 1:")
for fig in figures_fail:
    is_stub = fig.purpose == "Could not be parsed."
    print(f"  {fig.figure_id}: {'(STUB — LLM failed)' if is_stub else 'OK'}")
    print(f"    subfigures={len(fig.subfigures)}  datasets={fig.datasets_used}")

## 10. JSON round-trip verification

All `Figure` → JSON → `Figure` transformations must be lossless.

In [ ]:
json_str = fig.model_dump_json(indent=2)
fig_restored = Figure.model_validate_json(json_str)

assert fig_restored.figure_id == fig.figure_id
assert fig_restored.title == fig.title
assert len(fig_restored.subfigures) == len(fig.subfigures)
for orig, rest in zip(fig.subfigures, fig_restored.subfigures):
    assert orig.label == rest.label
    assert orig.plot_type == rest.plot_type

print(f"JSON round-trip: OK ({len(json_str):,} chars)")
print(f"\nFull JSON (first 800 chars):")
print(json_str[:800] + "...")

## 11. Live parse from PaperParser output (end-to-end demo)

Full pipeline: `PaperParser.parse()` → `FigureParser.parse_all_figures()`.

**Requires `ANTHROPIC_API_KEY` and network access.**

In [ ]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY not set — skipping live end-to-end demo.")
else:
    from researcher_ai.parsers.paper_parser import PaperParser

    paper_parser = PaperParser(cache_dir=CACHE_DIR)
    live_paper = paper_parser.parse("26971820")   # eCLIP paper

    print(f"Paper: {live_paper.title[:80]}")
    print(f"Figure IDs: {live_paper.figure_ids}\n")

    figure_parser = FigureParser(cache_dir=CACHE_DIR)
    live_figures = figure_parser.parse_all_figures(live_paper)

    for fig in live_figures:
        print(f"{'='*60}")
        print(f"{fig.figure_id}: {fig.title}")
        print(f"  Purpose:    {fig.purpose[:120]}...")
        print(f"  Subfigures: {len(fig.subfigures)}")
        for sf in fig.subfigures:
            print(f"    [{sf.label}] {sf.plot_type.value} — {sf.description[:55]}")
        print(f"  Datasets:   {fig.datasets_used}")
        print(f"  Methods:    {fig.methods_used}")

## Summary

Phase 3 delivers:

- **`FigureParser.parse_all_figures(paper)`** — one call, all figures extracted
- **`FigureParser.parse_figure(paper, figure_id)`** — single figure with full context
- **Caption extraction**: JATS dict first, regex section scan second, raw text fallback
- **In-text references**: regex sentence matching including sub-panel variants (`Fig. 1A-D`)
- **Dataset accessions**: regex-first (GSE, SRP, SRR, PXD, PRJNA, E-MTAB, EGAS…), LLM fallback
- **Subfigure decomposition** (LLM): `PlotType`, `Axis`, `ColorMapping`, `ErrorBarType`,
  `StatisticalAnnotation`, facet variable, individual points flag
- **Purpose extraction** (LLM): one-paragraph summary + short title
- **Method extraction** (LLM): assay and computational method names
- **Panel layout inference**: rows × cols + label style from subfigure count
- **Graceful failure**: per-figure stub on exception, batch continues
- **Full JSON round-trip** on the `Figure` model

Next: **Phase 4 — Methods Parser** (extract structured `Method` / `AssayGraph` objects).